# Batoid Pupil Comparison: Hexapod Defocus Configurations

**Author:** Aaron Roodman
**Date Created:** 2026-06-15
**Last Modified:** 2026-06-15
**Status:** In Progress
**Keywords:** batoid, pupil, donut, hexapod, M2, defocus, FAM, ray trace, spot diagram, AOS

## Description

Use **Batoid** + the LSST optical model to study how the giant-donut **pupil** depends on
*how* the defocus is produced, at the field position of our real giant donut on
**R30_S21** (pixel (1167, 2915)).

Method: trace a **dense grid of rays** (a spot diagram, many spots) from a point source
at that field angle through the defocused system to the detector; the 2-D **histogram of
ray positions is the geometric pupil/donut image**.

Comparisons:
1. **8 mm defocus, extra- and intra-focal:** Camera hexapod alone (±8 mm) vs Camera +
   M2 hexapods ±4 mm each (M2 is powered → contributes differently).
2. **FAM mode:** Camera hexapod ±1.5 mm — the intra- and extra-focal pupils used for
   curvature wavefront sensing.

Hexapod pistons are rigid z-shifts of the `LSSTCamera` group and `M2`
(`withGloballyShiftedOptic`; no `batoid_rubin` data download). Models: design
`LSST_r.yaml`, or as-built `Rubin_v3.12_r.yaml` / `Rubin_v3.14_r.yaml`.

**Output:** comparison figures; no files written by default.

**Based on:** the giant-donut study (`wfs_giant_donut_fit.ipynb`); R30_S21 seq 340
(intra_8mm) / 349 (extra_8mm).

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-15 | Aaron Roodman | Initial version: batoid spot-diagram pupil, camera-8mm vs camera-4mm+M2-4mm |
| 2026-06-15 | Aaron Roodman | Field angle from the R30_S21 donut pixel; add intra-focal + FAM (±1.5 mm) comparisons |
| 2026-06-15 | Aaron Roodman | FAM: invert the extra donut (x→−x, y→−y) so intra/extra overlay (residual std 2.08→1.18) |
| 2026-06-15 | Aaron Roodman | §7: extract danish's fit pupil (DonutFactory+SingleDonutModel from the ts_wep Instrument, z_fit=0, full off-axis z_ref) and overlay on the batoid donut (giant camera-only 8 mm + FAM 1.5 mm) with difference + radial profiles |
| 2026-06-15 | Aaron Roodman | §8 rim radius vs field angle (danish vs batoid, inner & outer); §9 per-surface vignetting attribution via batoid traceFull (off-axis loss dominated by M1/M2/M3 edges + filter) |
| 2026-06-15 | Aaron Roodman | §10 danish mask vs batoid vignetting boundary in the ts_wep stop-surface pupil frame: ~98.6% agree at R30; inner edge exact, outer edge non-circular and danish (circle) over-extends mean +16 / max +90 mm (filter, then M1) — the pupil-plane origin of the donut-fit rim residual |
| 2026-06-15 | Aaron Roodman | Default to design model LSST_r (= ts_wep's inst.batoidModelName); local-shift (withLocallyShiftedOptic) matching ts_wep; §10 rebuilt as the three-case (WFS Detector / FAM & giant Camera / cam+M2 split, intra+extra) danish-mask-vs-batoid comparison — WFS fixed mask is exact, mask mismatch is a camera-hexapod (filter/L1/L2) effect; add measured-M1-radius params |
| 2026-06-15 | Aaron Roodman | §11 per-element ellipse vs ts_wep circle edge fit: ellipse does NOT reduce the matched-config residual (M1 already circular) and over-clips far elements; a defocal-refit circle recovers giant-intra 96→99.5% — the fix is defocal-dependence, not ellipticity |
| 2026-06-16 | Aaron Roodman | §7: clearer danish-panel label — "no fitted aberration; z_ref = off-axis defocus" instead of "z_fit=0" |
| 2026-06-16 | Aaron Roodman | §10: note that the matched-config ~3 mm outer-edge residual is a finite-ray-grid artifact (scales with asPolar ring spacing, converges to ~0 with nrad), not a real mask error |
| 2026-06-16 | Aaron Roodman | §12 non-telecentricity (chief-ray AOI ~6.6°, intra/extra donut centroid shift); §13 synthetic-donut + ts_wep paired-danish-fit tooling and the M1M3 aperture-update Zernike-bias study (draw measured 2.5833/4.165, fit default 2.558/4.18) |
| 2026-06-16 | Aaron Roodman | Note §12 astrometric shift not germane (donut ellipticity ~7% dominated by off-axis mask, not cos-AOI foreshortening); §14 WFS R04 & §15 FAM R03 Danish/Batoid/Difference pupil grids 1.60–1.725°, intra+extra; §16 filter-vignetting onset; §17 pupil-mismatch Zernike-bias simulation summary |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Field Angle of the R30_S21 Donut](#field)
5. [8 mm Defocus: Camera vs Camera+M2 (intra & extra)](#eightmm)
6. [FAM Mode: Intra vs Extra (±1.5 mm)](#fam)
7. [Danish Pupil vs Batoid Spot Diagram](#danish)
8. [Rim Mismatch vs Field Angle](#rimfield)
9. [What Vignettes the Off-Axis Beam?](#vignframe)
10. [Danish Mask vs Batoid Vignetting Boundary — three defocal cases](#maskedge)
11. [Per-element ellipse edge model](#ellipse)
12. [Non-telecentricity in the Pupil/Donut Shape](#telecentric)
13. [Synthetic Donut + ts_wep Fit — M1M3 Aperture-Update Bias](#aperbias)
14. [WFS Pupil: Danish vs Batoid vs Difference](#wfsgrid)
15. [FAM Pupil: Danish vs Batoid vs Difference](#famgrid)
16. [Intra-focal Filter-Vignetting Onset](#filteronset)
17. [Pupil-Mismatch Zernike Bias from Simulation](#biassim)


<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# LSST optical model: "LSST_r.yaml" (design ~v3.3),
#   "Rubin_v3.12_r.yaml" (as-built, default), "Rubin_v3.14_r.yaml" (as-built).
# Design model (~v3.3): the model ts_wep/danish actually use
#   (inst.batoidModelName == "LSST_{band}"). As-built alternatives for the eventual
#   comparison to DATA: "Rubin_v3.12_r.yaml", "Rubin_v3.14_r.yaml".
model_yaml = "LSST_r.yaml"
wavelength = 620e-9              # metres (r-band ~620 nm)

# Field position: the real giant donut on R30_S21. theta is computed from this pixel
# (set theta_x_deg / theta_y_deg directly to override).
detector_name = "R30_S21"
donut_pixel = (1167, 2915)       # (x, y) px
theta_x_deg = None               # None -> from the pixel above
theta_y_deg = None

# Defocus magnitudes (mm)
defocus_mm = 8.0                 # camera-only defocus
m2_split_mm = 4.0                # split: camera + M2 each this much (-> 8 mm total move)
fam_offset_mm = 1.5              # FAM camera-hexapod offset (curvature sensing)

# Ray grid / histogram
n_pupil = 700                    # rays across the pupil grid (n_pupil^2 spots)
hist_npix = 256                  # output donut image size (pixels)
half_mm_8mm = 4.0                # half-width for the 8 mm donut panels (mm)
half_mm_fam = 1.0                # half-width for the FAM donut panels (mm)

cmap = "viridis"

# --- Defocal cases for the danish<->batoid comparison ---------------------------
# Each case applies z-pistons via withLocallyShiftedOptic (as ts_wep does). ts_wep
# offsets the Detector for the corner WFS (CCD piston, camera fixed) and the Camera
# hexapod for FAM / giant donuts (+ M2 for the unapportionable split).
wfs_offset_mm   = 1.5            # corner-WFS CCD piston
giant_offset_mm = defocus_mm     # giant-donut camera-hexapod piston (8 mm)
defocal_cases = [
    ("WFS extra",          [("Detector",    wfs_offset_mm)]),
    ("WFS intra",          [("Detector",   -wfs_offset_mm)]),
    ("FAM extra",          [("LSSTCamera",  fam_offset_mm)]),
    ("FAM intra",          [("LSSTCamera", -fam_offset_mm)]),
    ("giant extra",        [("LSSTCamera",  giant_offset_mm)]),
    ("giant intra",        [("LSSTCamera", -giant_offset_mm)]),
    ("giant cam+M2 extra", [("LSSTCamera",  giant_offset_mm / 2), ("M2",  giant_offset_mm / 2)]),
    ("giant cam+M2 intra", [("LSSTCamera", -giant_offset_mm / 2), ("M2", -giant_offset_mm / 2)]),
]

# Newly measured M1 aperture radii (m). NOT applied to the batoid<->danish comparison
# (each model must use its own values for self-consistency); recorded for the eventual
# comparison to DATA, where danish's mask radii should be updated to these.
m1_outer_measured = 4.165        # current design/danish value: 4.18
m1_inner_measured = 2.5833       # current design/danish value: 2.558

# --- Synthetic-donut + ts_wep-fit studies (§12 non-telecentricity, §13 aperture bias) ---
th_fov          = 1.72           # field radius (deg) for the donut studies (just inside the 1.725 cut)
seeing_arcsec   = 0.7            # Gaussian seeing FWHM applied to the batoid spot diagram
donut_peak_e    = 1.0e4          # peak electrons/pixel in the synthetic donut
donut_sky_e     = 200.0          # sky pedestal (e/pix) so the fit has a finite background
fit_noll        = list(range(4, 23))   # Noll indices fit by danish (Z4..Z22)


<a id='setup'></a>
## Setup & Imports

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import batoid

import lsst.geom
from lsst.afw.cameraGeom import FIELD_ANGLE, PIXELS
from lsst.obs.lsst import LsstCam

sys.path.insert(0, str(Path.cwd().parent))
from common.utils import setup_plotting

setup_plotting()
camera = LsstCam.getCamera()


<a id='functions'></a>
## Helper Functions

In [ ]:
def field_angle_from_pixel(detector_name, x, y):
    """Field angle (deg) for a detector pixel, from LSST camera geometry.

    Note: gives the correct field *radius*; the absolute orientation vs batoid's
    field-angle frame is approximate (fine for comparing configs at one field point).
    """
    fa = camera[detector_name].getTransform(PIXELS, FIELD_ANGLE).applyForward(
        lsst.geom.Point2D(x, y))
    return float(np.degrees(fa.getX())), float(np.degrees(fa.getY()))


def build_case(fiducial, offsets):
    """Apply a list of (optic_name, dz_mm) z-pistons via withLocallyShiftedOptic — the same
    call ts_wep uses for the defocal model. Detector piston = corner-WFS; LSSTCamera piston
    = FAM / giant; add M2 for the camera+M2 split."""
    tel = fiducial
    for optic, dz in offsets:
        tel = tel.withLocallyShiftedOptic(optic, [0, 0, dz * 1e-3])
    return tel


def build_telescope(fiducial, camera_dz_mm, m2_dz_mm):
    """Camera- and M2-hexapod z pistons (signed mm), via build_case (local shifts).
    +z = extra-focal sense, -z = intra-focal (per the chosen convention)."""
    offs = []
    if camera_dz_mm:
        offs.append(("LSSTCamera", camera_dz_mm))
    if m2_dz_mm:
        offs.append(("M2", m2_dz_mm))
    return build_case(fiducial, offs)


def trace_donut(tel, theta_x_deg, theta_y_deg, wavelength, n_pupil, npix, half_mm):
    """Trace a dense pupil ray grid; histogram (un-vignetted) positions -> donut image,
    centred on the donut centroid. Returns dict(image, span_mm, n_rays)."""
    rays = batoid.RayVector.asGrid(
        optic=tel, wavelength=wavelength,
        theta_x=np.deg2rad(theta_x_deg), theta_y=np.deg2rad(theta_y_deg), nx=n_pupil)
    tel.trace(rays)
    ok = ~rays.vignetted & ~rays.failed
    x = rays.x[ok] * 1e3; y = rays.y[ok] * 1e3
    cx, cy = x.mean(), y.mean()
    H, _, _ = np.histogram2d(x, y, bins=npix,
                             range=[[cx - half_mm, cx + half_mm],
                                    [cy - half_mm, cy + half_mm]])
    return {"image": H.T, "span_mm": float(x.max() - x.min()), "n_rays": int(ok.sum())}


def trace_configs(fiducial, configs, theta, n_pupil, npix, half_mm):
    """configs: list of (label, camera_dz_mm, m2_dz_mm). Returns list of (label, donut)."""
    out = []
    for label, cdz, mdz in configs:
        tel = build_telescope(fiducial, cdz, mdz)
        out.append((label, trace_donut(tel, theta[0], theta[1], wavelength,
                                        n_pupil, npix, half_mm)))
    return out


def azimuthal_profile(image, half_mm, nbin=120):
    """Azimuthally averaged radial profile of a centred donut image -> (r_mm, value)."""
    n = image.shape[0]
    yy, xx = np.mgrid[0:n, 0:n]
    c = (n - 1) / 2.0
    r = np.hypot(xx - c, yy - c) * (2 * half_mm / n)
    edges = np.linspace(0, half_mm, nbin + 1)
    idx = np.digitize(r.ravel(), edges) - 1
    v = image.ravel()
    rc, prof = [], []
    for k in range(nbin):
        sel = idx == k
        if sel.any():
            rc.append(0.5 * (edges[k] + edges[k + 1])); prof.append(v[sel].mean())
    return np.array(rc), np.array(prof)


def show_pair_with_diff(ax_a, ax_b, ax_d, da, db, half_mm, labels):
    """Plot donut A, donut B, and A-B on three axes."""
    vmax = max(da["image"].max(), db["image"].max())
    ext = [-half_mm, half_mm, -half_mm, half_mm]
    for ax, d, lab in ((ax_a, da, labels[0]), (ax_b, db, labels[1])):
        ax.imshow(d["image"], origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
        ax.set_title(f"{lab}\nspan {d['span_mm']:.2f} mm", fontsize=9)
        ax.set_xlabel("mm"); ax.set_ylabel("mm")
    diff = da["image"] - db["image"]
    v = np.nanpercentile(np.abs(diff), 99.5)
    im = ax_d.imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=ext)
    ax_d.set_title(f"({labels[0]}) − ({labels[1]})", fontsize=9)
    ax_d.set_xlabel("mm"); ax_d.set_ylabel("mm")
    return im


<a id='field'></a>
## Field Angle of the R30_S21 Donut

In [ ]:
if theta_x_deg is None or theta_y_deg is None:
    tx, ty = field_angle_from_pixel(detector_name, *donut_pixel)
else:
    tx, ty = theta_x_deg, theta_y_deg
theta = (tx, ty)
print(f"{detector_name} pixel {donut_pixel}: field angle = ({tx:.4f}, {ty:.4f}) deg, "
      f"radius = {np.hypot(tx, ty):.4f} deg")
print(f"model: {model_yaml}")


<a id='eightmm'></a>
## 8 mm Defocus: Camera vs Camera+M2 (intra & extra)

In [ ]:
fiducial = batoid.Optic.fromYaml(model_yaml)

configs_8mm = [
    ("extra: cam +8",            +defocus_mm, 0.0),
    ("extra: cam +4 + M2 +4",    +m2_split_mm, +m2_split_mm),
    ("intra: cam -8",            -defocus_mm, 0.0),
    ("intra: cam -4 + M2 -4",    -m2_split_mm, -m2_split_mm),
]
d8 = dict(trace_configs(fiducial, configs_8mm, theta, n_pupil, hist_npix, half_mm_8mm))
for lab, d in d8.items():
    print(f"  {lab:>24}: span {d['span_mm']:.2f} mm, {d['n_rays']} rays")

# Rows: extra, intra.  Cols: camera-only, camera+M2, difference.
fig, axes = plt.subplots(2, 3, figsize=(14, 9), constrained_layout=True)
show_pair_with_diff(axes[0, 0], axes[0, 1], axes[0, 2],
                    d8["extra: cam +8"], d8["extra: cam +4 + M2 +4"], half_mm_8mm,
                    ("extra: cam +8", "extra: cam +4 + M2 +4"))
show_pair_with_diff(axes[1, 0], axes[1, 1], axes[1, 2],
                    d8["intra: cam -8"], d8["intra: cam -4 + M2 -4"], half_mm_8mm,
                    ("intra: cam -8", "intra: cam -4 + M2 -4"))
fig.suptitle(f"8 mm defocus pupil — {model_yaml}, field ({tx:.2f}, {ty:.2f})°  "
             f"[camera-only vs camera+M2]", fontsize=12)
plt.show()


In [ ]:
# Azimuthally averaged radial profiles for the 8 mm configs.
fig, axs = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for ax, side in zip(axs, ("extra", "intra")):
    for lab, d in d8.items():
        if lab.startswith(side):
            r, p = azimuthal_profile(d["image"], half_mm_8mm)
            ax.plot(r, p, label=lab)
    ax.set_xlabel("radius from centre [mm]"); ax.set_ylabel("rays / pixel")
    ax.set_title(f"{side}-focal radial profile"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.show()


<a id='fam'></a>
## FAM Mode: Intra vs Extra (±1.5 mm)

In [ ]:
configs_fam = [
    ("FAM intra: cam -1.5", -fam_offset_mm, 0.0),
    ("FAM extra: cam +1.5", +fam_offset_mm, 0.0),
]
dfam = dict(trace_configs(fiducial, configs_fam, theta, n_pupil, hist_npix, half_mm_fam))
for lab, d in dfam.items():
    print(f"  {lab:>22}: span {d['span_mm']:.2f} mm, {d['n_rays']} rays")

# Intra and extra are related by a parity flip, so invert the extra donut
# (x -> -x, y -> -y, i.e. a 180° point reflection of the centred image) to overlay it
# on the intra donut. The difference then shows the genuine intra/extra mismatch.
extra = dfam["FAM extra: cam +1.5"]
extra_inv = {"image": extra["image"][::-1, ::-1], "span_mm": extra["span_mm"]}

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)
show_pair_with_diff(axes[0], axes[1], axes[2],
                    dfam["FAM intra: cam -1.5"], extra_inv, half_mm_fam,
                    ("FAM intra: cam -1.5", "FAM extra: cam +1.5 (inverted x,y)"))
fig.suptitle(f"FAM-mode pupil (camera ±{fam_offset_mm} mm) — {model_yaml}, "
             f"field ({tx:.2f}, {ty:.2f})°  [extra inverted to match intra]", fontsize=11)
plt.show()

# Radial profiles are azimuthal averages (flip-invariant), so compare directly.
fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
for lab, d in dfam.items():
    r, p = azimuthal_profile(d["image"], half_mm_fam)
    ax.plot(r, p, label=lab)
ax.set_xlabel("radius from centre [mm]"); ax.set_ylabel("rays / pixel")
ax.set_title("FAM intra vs extra radial profile"); ax.grid(alpha=0.3); ax.legend()
plt.show()


<a id='danish'></a>
## Danish Pupil vs Batoid Spot Diagram

Extract the **pupil danish actually uses in its fit** and overlay it on the batoid
spot-diagram donut. danish has no method to return the pupil, so we build its
`DonutFactory` + `SingleDonutModel` from the **ts_wep `Instrument`** (pupil obscuration
`R_outer`/`R_inner`, `mask_params` for M1/M2/M3/L1/Filter/spider, focal length), set
`z_ref` to the instrument **off-axis Zernikes** (which carry the defocus + design
off-axis aberration), use **no fitted Zernikes** (`z_fit=()`) and a tiny `fwhm` (0.3″),
and draw — giving danish's geometric donut/pupil.

Per **Josh Meyers**, danish/ts_wep assume the giant-donut defocus is **camera-only**, so
the danish giant pupil is compared to the batoid **camera-only +8 mm** donut (not the
camera-4 + M2-4 that physically produced the data — that mismatch is the A−B difference
in §5). danish and batoid share the field-angle convention, so the donuts register
directly (identity, no flip). Both images are unit-sum normalised; the difference shows
where danish's parametric pupil departs from batoid's full ray trace (vignetting/baffles).


In [ ]:
import danish
from lsst.ts.wep.utils import getTaskInstrument, DefocalType

compare_npix = 255   # danish requires odd npix; common grid for danish & batoid
danish_fwhm = 0.3    # arcsec; tiny blur -> sharp danish pupil edges


def danish_pupil_donut(detector_name, defocal_mm, defocal_type, theta_deg, npix,
                       half_mm, fwhm=0.3):
    """Geometric donut/pupil danish uses in its fit, at the given defocus, on a ±half_mm
    grid (unit-sum normalised). Builds danish's DonutFactory + SingleDonutModel from the
    ts_wep Instrument; z_ref = off-axis coeffs (carry defocus + design off-axis), and no
    fitted Zernikes (z_fit=())."""
    inst = getTaskInstrument("LSSTCam", detector_name)
    inst.defocalOffset = defocal_mm * 1e-3
    txd, tyd = theta_deg
    z_ref = inst.getOffAxisCoeff(txd, tyd, defocal_type, "r",
                                 nollIndicesModel=np.arange(79),
                                 nollIndicesIntr=np.arange(4, 23))
    pscale = 2 * half_mm * 1e-3 / npix          # m/pixel so the image spans ±half_mm
    factory = danish.DonutFactory(
        R_outer=inst.radius, R_inner=inst.radius * inst.obscuration,
        mask_params=inst.maskParams, focal_length=inst.focalLength, pixel_scale=pscale)
    model = danish.SingleDonutModel(factory, z_ref=z_ref, z_terms=(),
                                    thx=np.deg2rad(txd), thy=np.deg2rad(tyd), npix=npix)
    img = model.model(flux=1.0, dx=0.0, dy=0.0, fwhm=fwhm, z_fit=())
    return img / img.sum()


def batoid_norm(cam_dz_mm, m2_dz_mm, half_mm, npix):
    """Batoid spot-diagram donut on a ±half_mm grid, unit-sum normalised."""
    tel = build_telescope(fiducial, cam_dz_mm, m2_dz_mm)
    b = trace_donut(tel, tx, ty, wavelength, n_pupil, npix, half_mm)
    return b["image"] / b["image"].sum()


# danish assumes CAMERA-ONLY defocus -> compare to batoid camera-only.
comparisons = [
    ("giant: camera-only +8 mm", defocus_mm,   DefocalType.Extra, half_mm_8mm),
    ("FAM: camera +1.5 mm",      fam_offset_mm, DefocalType.Extra, half_mm_fam),
]
results = []
for label, cdz, dtype, half in comparisons:
    B = batoid_norm(cdz, 0.0, half, compare_npix)
    D = danish_pupil_donut(detector_name, cdz, dtype, theta, compare_npix, half, danish_fwhm)
    results.append((label, half, B, D))

fig, axes = plt.subplots(len(results), 3, figsize=(13, 4.6 * len(results)),
                         constrained_layout=True)
axes = np.atleast_2d(axes)
for row, (label, half, B, D) in enumerate(results):
    ext = [-half, half, -half, half]; vmax = max(B.max(), D.max())
    axes[row, 0].imshow(B, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
    axes[row, 0].set_title(f"batoid — {label}", fontsize=9)
    axes[row, 1].imshow(D, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
    axes[row, 1].set_title("danish pupil (no fitted aberration;\nz_ref = off-axis defocus)", fontsize=8)
    diff = B - D; v = np.nanpercentile(np.abs(diff), 99.5)
    axes[row, 2].imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=ext)
    axes[row, 2].set_title("batoid − danish", fontsize=9)
    for a in axes[row]:
        a.set_xlabel("mm"); a.set_ylabel("mm")
fig.suptitle(f"Danish pupil vs batoid spot diagram — {model_yaml}, "
             f"field ({tx:.2f}, {ty:.2f})°", fontsize=12)
plt.show()

# Azimuthally averaged radial profiles (orientation-independent).
fig, axs = plt.subplots(1, len(results), figsize=(7 * len(results), 4.4),
                        constrained_layout=True)
axs = np.atleast_1d(axs)
for ax, (label, half, B, D) in zip(axs, results):
    rB, pB = azimuthal_profile(B, half); rD, pD = azimuthal_profile(D, half)
    ax.plot(rB, pB, label="batoid"); ax.plot(rD, pD, label="danish")
    ax.set_title(label); ax.set_xlabel("radius [mm]"); ax.set_ylabel("norm. illumination")
    ax.grid(alpha=0.3); ax.legend()
plt.show()


<a id='rimfield'></a>
## Rim Mismatch vs Field Angle

The danish and batoid pupils agree to within a pixel on-axis but **diverge off-axis** — and
that field-dependent rim mismatch is the prime suspect for the inner/outer-rim residuals
seen in the WFS/FAM donut fits (which all sit at field positions, the corners most of all).

We sweep the field radius from 0 to the R30_S21 edge (~1.76°, along its azimuth) and
measure the **inner and outer rim radii** (half-max of the azimuthal profile, sub-bin
interpolated) of the batoid spot-diagram donut and danish's fit pupil, for both the giant
(camera +8 mm) and FAM (camera +1.5 mm) camera-only defocus. The bottom row is the
**danish − batoid** rim difference in µm: ~0 to ~0.6°, then growing to a few tens of µm
at the field edge.

In [ ]:
# ------------------------------------------------------------------
# Rim radius vs field angle: where does danish depart from batoid?
# (depends on the §7 helpers danish_pupil_donut / compare_npix / danish_fwhm)
# ------------------------------------------------------------------
def _edge_crossings(r, p):
    """Inner & outer half-max radii of a centred donut radial profile, linearly
    interpolated between bins for sub-bin precision."""
    hm = 0.5 * p.max()
    above = np.where(p > hm)[0]
    if len(above) == 0:
        return np.nan, np.nan
    i0, i1 = above.min(), above.max()

    def cross(ia, ib):                       # interpolate to p == hm between bins ia, ib
        if ia < 0 or ib >= len(p) or p[ib] == p[ia]:
            return r[np.clip(ia if ia >= 0 else ib, 0, len(r) - 1)]
        return r[ia] + (hm - p[ia]) * (r[ib] - r[ia]) / (p[ib] - p[ia])

    return cross(i0 - 1, i0), cross(i1 + 1, i1)   # rising (inner), falling (outer)


def rim_vs_field(cam_dz_mm, dtype, half_mm, fields_deg, nbin=220):
    """Inner & outer rim radii of the batoid and danish camera-only donuts at each field
    radius (along the R30_S21 azimuth). Returns rows of (field_deg, bi, bo, di, do) mm."""
    az = np.arctan2(ty, tx)                  # R30_S21 azimuth
    rows = []
    for R in fields_deg:
        txd, tyd = R * np.cos(az), R * np.sin(az)
        tel = build_telescope(fiducial, cam_dz_mm, 0.0)
        Bimg = trace_donut(tel, txd, tyd, wavelength, n_pupil, hist_npix, half_mm)["image"]
        Dimg = danish_pupil_donut(detector_name, cam_dz_mm, dtype, (txd, tyd),
                                  compare_npix, half_mm, danish_fwhm)
        bi, bo = _edge_crossings(*azimuthal_profile(Bimg, half_mm, nbin=nbin))
        di, do = _edge_crossings(*azimuthal_profile(Dimg, half_mm, nbin=nbin))
        rows.append((R, bi, bo, di, do))
    return np.array(rows)


sweep_fields = np.array([0.0, 0.3, 0.6, 0.9, 1.2, 1.5, 1.76])
sweep = {
    "giant (camera +8 mm)": rim_vs_field(defocus_mm,   DefocalType.Extra, half_mm_8mm, sweep_fields),
    "FAM (camera +1.5 mm)": rim_vs_field(fam_offset_mm, DefocalType.Extra, half_mm_fam, sweep_fields),
}

fig, axes = plt.subplots(2, len(sweep), figsize=(7 * len(sweep), 8.5),
                         constrained_layout=True)
for col, (label, S) in enumerate(sweep.items()):
    R, bi, bo, di, do = S.T
    ax = axes[0, col]
    ax.plot(R, bi, "o-",  color="C0", label="batoid inner")
    ax.plot(R, bo, "o-",  color="C1", label="batoid outer")
    ax.plot(R, di, "s--", color="C0", mfc="none", label="danish inner")
    ax.plot(R, do, "s--", color="C1", mfc="none", label="danish outer")
    ax.set_title(label); ax.set_xlabel("field radius [deg]"); ax.set_ylabel("rim radius [mm]")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
    axd = axes[1, col]
    axd.axhline(0, color="k", lw=0.6)
    axd.plot(R, (di - bi) * 1e3, "s-", color="C0", label="inner Δ")
    axd.plot(R, (do - bo) * 1e3, "s-", color="C1", label="outer Δ")
    axd.set_xlabel("field radius [deg]"); axd.set_ylabel("danish − batoid [µm]")
    axd.grid(alpha=0.3); axd.legend(fontsize=8)
fig.suptitle("Pupil rim radius vs field angle (camera-only defocus, R30_S21 azimuth)",
             fontsize=12)
plt.show()

for label, S in sweep.items():
    R, bi, bo, di, do = S.T
    print(f"{label}: at {R[-1]:.2f}°  inner Δ={1e3*(di[-1]-bi[-1]):+.1f} µm, "
          f"outer Δ={1e3*(do[-1]-bo[-1]):+.1f} µm   (on-axis Δ≈0)")


<a id='vignframe'></a>
## What Vignettes the Off-Axis Beam?

To find the *physical source* of the off-axis rim mismatch, ray-trace the full pupil grid
with batoid `traceFull` and record, for each ray, the **first surface that vignettes it**.
Counting losses among the rays that an ideal annular pupil `[R_in, R_out]` would pass
isolates the field-dependent vignetting batoid captures but danish's parametric mask only
approximates.

On-axis nothing in the annulus is lost; off-axis at R30_S21 a large fraction is clipped —
dominated by the **M1/M2/M3 mirror edges** (the beam footprint walks off the mirrors), with
a smaller **filter** contribution. This is exactly the field-dependent aperture that, if
imperfectly captured in danish's pupil mask, leaves the rim residuals in the fit.

In [ ]:
# ------------------------------------------------------------------
# Which optical surface vignettes the off-axis beam?  (batoid traceFull)
# ------------------------------------------------------------------
import collections


def vignetting_breakdown(cam_dz_mm, theta_deg, nx=500):
    """Fraction of nominal-annulus rays vignetted, and the per-surface breakdown, for a
    camera-only defocus at the given field. Returns (total_frac, {surface: frac})."""
    inst = getTaskInstrument("LSSTCam", detector_name)
    R_out, R_in = inst.radius, inst.radius * inst.obscuration
    tel = build_telescope(fiducial, cam_dz_mm, 0.0)
    rays = batoid.RayVector.asGrid(optic=tel, wavelength=wavelength,
                                   theta_x=np.deg2rad(theta_deg[0]),
                                   theta_y=np.deg2rad(theta_deg[1]), nx=nx)
    pr = np.hypot(rays.x, rays.y)            # entrance-pupil radius (m)
    first = np.array([""] * len(pr), dtype=object)
    vprev = np.zeros(len(pr), bool)
    for name, d in tel.traceFull(rays).items():
        vo = d["out"].vignetted
        newly = vo & ~vprev
        first[newly] = name
        vprev = vprev | vo
    ann = (pr >= R_in) & (pr <= R_out)       # rays an ideal annulus would pass
    lost = ann & vprev
    n_ann = int(ann.sum())
    frac = {s: int((lost & (first == s)).sum()) / n_ann
            for s in collections.Counter(first[lost])}
    return lost.sum() / n_ann, dict(sorted(frac.items(), key=lambda kv: -kv[1]))


R_edge = np.hypot(tx, ty)
cases = [("on-axis", (0.0, 0.0)), (f"R30_S21 ({R_edge:.2f}°)", (tx, ty))]
print("Giant camera-only +8 mm — vignetting of nominal-annulus rays:")
breakdowns = []
for lbl, th in cases:
    tot, frac = vignetting_breakdown(defocus_mm, th)
    breakdowns.append((lbl, tot, frac))
    print(f"\n  {lbl}: {100*tot:.1f}% of annulus rays vignetted")
    for s, f in frac.items():
        print(f"     {s:<16s} {100*f:5.1f}%")

lbl, tot, frac = breakdowns[-1]              # the off-axis case
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
names = list(frac.keys()); vals = [100 * frac[n] for n in names]
bars = ax.bar(names, vals, color="C3")
ax.set_ylabel("% of nominal-annulus rays vignetted")
ax.set_title(f"Off-axis vignetting by surface — giant camera +8 mm, {lbl}\n"
             f"total {100*tot:.1f}%  (on-axis: 0%)")
ax.tick_params(axis="x", rotation=30)
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width() / 2, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)
plt.show()


<a id='maskedge'></a>
## Danish Mask vs Batoid Vignetting Boundary — three defocal cases

danish's `maskParams` are circles ts_wep fits to the batoid vignetting boundary (one per
optical element, cubic-in-θ centre & radius; `maskUtils._fitEdges`). They are a **single
fixed config** shared by every instrument. ts_wep does, however, model the defocal
configuration correctly when it builds the off-axis `z_ref`: it pistons the **Detector**
for the corner WFS (CCD offset, camera fixed) and the **Camera hexapod** for FAM / giant
donuts (`batoidOffsetOptic`). So the question is whether that *fixed* mask still matches
the true boundary in each case.

We sample the pupil in ts_wep's frame (rays on `optic.stopSurface`), offset the same optic
ts_wep does (`withLocallyShiftedOptic`), and compare the fixed danish mask to the batoid
boundary for all three cases — WFS (Detector ±1.5 mm), FAM (Camera ±1.5 mm), giant (Camera
±8 mm, and the 4 mm + 4 mm camera+M2 split), both intra and extra — on the **design model**
(`LSST_r`, = `inst.getBatoidModel()`).

**Result.** For the **WFS the fixed mask is exactly right** (camera fixed → boundary =
in-focus mask for both signs; only the small ~3 mm M1 circle-shape residual). The mask
mismatch is entirely a **camera-hexapod** effect — the filter/L1/L2 ride with the camera,
so off-axis they clip the pupil differently for intra vs extra, which the fixed mask cannot
follow: small at FAM ±1.5 mm (intra outer edge over-extends up to ~+57 mm via the filter),
large at giant ±8 mm (intra up to ~+261 mm), with the extra/intra signs opposite. The
camera+M2 split gives yet another pupil that ts_wep cannot represent.

*Note — the in-focus / WFS baseline (`outerΔ` ~+3 mm mean, +8 mm max; the `M1 699` over-extend) is a **finite-ray-grid artifact**, not a real mask error: it scales with the `asPolar` radial ring spacing (annulus/`nrad` = 8.1 mm at `nrad=200`) and converges toward 0 with finer sampling (+3.1 → +1.0 mm for `nrad` 200 → 1600). The outermost sampled ring on the M1 rim is counted vignetted by the trace, one ring inside danish's inclusive 4.18 m circle.*

In [ ]:
# ------------------------------------------------------------------
# Three-case danish-mask vs batoid vignetting boundary (design model).
#   maskParams = one fixed config (same for WFS & FAM instruments); compare it to the
#   batoid boundary for each defocal case, offsetting the optic ts_wep offsets.
# ------------------------------------------------------------------
import collections

inst_mask = getTaskInstrument("LSSTCam", detector_name)   # maskParams identical across dets
mask_params = inst_mask.maskParams
r_pup = inst_mask.radius
thr = np.hypot(tx, ty)


def danish_mask_circles(thx_deg, thy_deg):
    """Active danish mask circles at a field angle (ts_wep maskUtils: polyval in θ_deg,
    centre along the field direction): list of (surface, edge, clear, cx, cy, r)."""
    th = np.hypot(thx_deg, thy_deg)
    out = []
    for surf, edges in mask_params.items():
        if surf == "Spider_3D":
            continue
        for edge, p in edges.items():
            if th < p["thetaMin"] or th > p["thetaMax"]:
                continue
            r = np.polyval(p["radius"], th); c = np.polyval(p["center"], th)
            out.append((surf, edge, p["clear"], c * thx_deg / th, c * thy_deg / th, r))
    return out


def danish_pupil_mask(u, v, circles):
    keep = np.ones(len(u), bool)
    for surf, edge, clear, cx, cy, r in circles:
        inside = ((u - cx) ** 2 + (v - cy) ** 2) <= r ** 2
        keep &= inside if clear else ~inside
    return keep


def stop_boundary(optic, thx_deg, thy_deg, nrad=200, naz=1400):
    """Pupil sample on optic.stopSurface with the final vignetting flag and first
    vignetting surface (ts_wep's mask-fitting frame)."""
    rays = batoid.RayVector.asPolar(optic=optic, wavelength=wavelength,
                                    theta_x=np.deg2rad(thx_deg), theta_y=np.deg2rad(thy_deg),
                                    nrad=nrad, naz=naz)
    pr = rays.copy().toCoordSys(optic.stopSurface.coordSys)
    optic.stopSurface.surface.intersect(pr)
    xP, yP = pr.x.copy(), pr.y.copy()
    first = np.array([""] * len(xP), dtype=object); vprev = np.zeros(len(xP), bool)
    for name, d in optic.traceFull(rays).items():
        vo = d["out"].vignetted; newly = vo & ~vprev
        first[newly] = name; vprev = vprev | vo
    return xP, yP, ~vprev, first


circ = danish_mask_circles(tx, ty)
case_results = []
for label, offsets in defocal_cases:
    tel = build_case(fiducial, offsets)
    xP, yP, kept, first = stop_boundary(tel, tx, ty)
    dk = danish_pupil_mask(xP, yP, circ)
    perm = dk & ~kept                                  # danish keeps, batoid vignettes
    rr = np.hypot(xP, yP); ph = np.degrees(np.arctan2(yP, xP))
    nb_az = 72; be = np.linspace(-180, 180, nb_az + 1); pcen = 0.5 * (be[:-1] + be[1:])
    bo, do = [], []
    for k in range(nb_az):
        m = (ph >= be[k]) & (ph < be[k + 1]); bk = m & kept; dkk = m & dk
        bo.append(rr[bk].max() if bk.sum() > 3 else np.nan)
        do.append(rr[dkk].max() if dkk.sum() > 3 else np.nan)
    bo, do = np.array(bo), np.array(do)
    case_results.append(dict(
        label=label, agree=(dk == kept).mean(), perm=int(perm.sum()),
        by_surf=dict(collections.Counter(first[perm]).most_common(3)),
        outer_mean=1e3 * np.nanmean(do - bo), outer_max=1e3 * np.nanmax(do - bo),
        pcen=pcen, bo=bo, do=do))

print(f"Danish fixed mask vs batoid boundary at {detector_name} ({thr:.2f}°), model {model_yaml}")
print(f"{'case':<22s}{'agree':>8s}{'outerΔmean':>12s}{'max[mm]':>9s}   over-extend (by surface)")
for r in case_results:
    print(f"{r['label']:<22s}{100*r['agree']:7.2f}%{r['outer_mean']:>11.0f}{r['outer_max']:>9.0f}   "
          f"{r['perm']:5d} {r['by_surf']}")


In [ ]:
# Summary: outer-edge error per case, and error-vs-azimuth for one case per regime.
labels = [r["label"] for r in case_results]
x = np.arange(len(labels))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
a1.bar(x - 0.2, [r["outer_mean"] for r in case_results], 0.4, label="outer Δ mean")
a1.bar(x + 0.2, [r["outer_max"] for r in case_results], 0.4, label="outer Δ max")
a1.axhline(0, color="k", lw=0.6)
a1.set_xticks(x); a1.set_xticklabels(labels, rotation=40, ha="right", fontsize=7)
a1.set_ylabel("danish − batoid outer edge [mm]")
a1.set_title("Fixed-mask outer-edge error by defocal case"); a1.legend(fontsize=8); a1.grid(alpha=0.3, axis="y")
for r in case_results:
    if r["label"] in ("WFS intra", "FAM intra", "giant intra"):
        a2.plot(r["pcen"], (r["do"] - r["bo"]) * 1e3, "o-", ms=3, label=r["label"])
a2.axhline(0, color="k", lw=0.6)
a2.set_xlabel("pupil azimuth [deg]"); a2.set_ylabel("danish − batoid outer edge [mm]")
a2.set_title("Outer-edge error vs azimuth (intra)\nWFS flat ≈ 0; camera cases diverge (filter)")
a2.grid(alpha=0.3); a2.legend(fontsize=8)
plt.show()


<a id='ellipse'></a>
## Per-element ellipse edge model — does it reduce the residual?

ts_wep's `maskUtils._fitCircle` fits a **2-parameter circle** (centre on the field axis,
`(xc, r)`) to each element's projected edge (`asPolar` with `theta_y=0`, top half +
reflection). A circular aperture viewed off-axis projects to an **ellipse**, so the natural
generalization is a **3-parameter ellipse** `(xc, a, b)` — centre on the field axis, semi-axis
`a` along the field and `b` across it. Here we fit both to every element edge and ask whether
the ellipse reduces (a) the **matched-config** residual and (b) the **giant-intra** residual,
versus simply **refitting at the defocal configuration**.

**Result — the ellipse is not the fix.** At matched (in-focus) config the ellipse leaves the
outer-edge residual unchanged (~+2.4 mm): that residual is dominated by **M1, which is already
perfectly circular** (`a = b = 4.18`), so there is nothing for an ellipse to improve. Worse,
the far camera-borne elements (filter/L1/L2) only present a short, strongly-curved edge arc on
the pupil; a *closed* ellipse fit to that arc **over-clips** the pupil and lowers the overall
agreement. The decisive lever is **defocal-dependence**: refitting the mask at the camera
position recovers giant-intra from 96% to **99.5% with a plain circle** — the ellipse adds
nothing. So for danish the recommendation is to make the mask **defocal-configuration-aware**
(refit / shift the camera-borne element edges with the hexapod), keeping the circle model;
an ellipse per element is not worth the extra parameter.

In [ ]:
# ------------------------------------------------------------------
# §11  Per-element ellipse vs ts_wep's circle edge fit.
#   ts_wep _fitCircle: 2-param circle (centre on field axis) to each projected edge.
#   Ellipse generalization: 3-param (xc, a, b), a along / b across the field.
# ------------------------------------------------------------------
from scipy.optimize import least_squares
from scipy.interpolate import griddata
from sklearn.neighbors import NearestNeighbors

phi = np.arctan2(ty, tx)


def _edge_pupil_pts(xP, yP, xI, yI, rEdge):
    """Pupil points on an element's projected edge (faithful to ts_wep _fitCircle)."""
    xC = (xI.max() + xI.min()) / 2; xI = xI - xC
    xR = (xI.max() - xI.min()) / 2; yR = (yI.max() - yI.min()) / 2
    dAz = np.diff(yI[:2]) / xR
    az = np.arange(0, np.pi, abs(dAz[0]) if dAz.size and dAz[0] != 0 else 0.01)
    xE = rEdge * np.cos(az) - xC; yE = rEdge * np.sin(az)
    idx = np.where((xE / xR) ** 2 + (yE / yR) ** 2 <= 1); xE, yE = xE[idx], yE[idx]
    if xE.size < 5:
        return None
    iP = np.array([xI, yI]).T; pP = np.array([xP, yP]).T; eP = np.array([xE, yE]).T
    nbrs = NearestNeighbors(n_neighbors=3).fit(iP)
    nI = np.unique(nbrs.kneighbors(eP, return_distance=False))
    xq = griddata(iP[nI], pP[nI, 0], eP); yq = griddata(iP[nI], pP[nI, 1], eP)
    m = np.isfinite(xq) & np.isfinite(yq)
    if m.sum() < 5:
        return None
    xq, yq = xq[m], yq[m]
    return np.concatenate((xq[::-1], xq)), np.concatenate((yq[::-1], -yq))


def _fit_circle(x, y):
    return least_squares(lambda p: (x - p[0]) ** 2 + y ** 2 - p[1] ** 2, (-1, 5),
                         bounds=((-np.inf, 0), (0, np.inf))).x


def _fit_ellipse(x, y):
    return least_squares(lambda p: ((x - p[0]) / p[1]) ** 2 + (y / p[2]) ** 2 - 1, (-1, 4, 4),
                         bounds=((-np.inf, 1e-3, 1e-3), (0, np.inf, np.inf))).x


def fit_edges(optic):
    """Fit a circle (xc, r) and ellipse (xc, a, b) to each element edge crossing the pupil,
    at field radius thr (theta_y=0), the way ts_wep maskUtils does."""
    rPo = optic.pupilSize / 2
    rays = batoid.RayVector.asPolar(optic, theta_x=np.deg2rad(thr), theta_y=0,
                                    wavelength=wavelength, inner=0, outer=1.1 * rPo,
                                    nrad=100, naz=int(2 * np.pi * 100))
    tr = optic.traceFull(rays.copy())
    pr = rays.toCoordSys(optic.stopSurface.coordSys); optic.stopSurface.surface.intersect(pr)
    xP, yP = pr.x, pr.y; fits = {}
    for name in tr:
        it = optic.itemDict[optic._names[name]]
        if isinstance(it, batoid.Detector):
            continue
        xI, yI = tr[name]["out"].x, tr[name]["out"].y
        if isinstance(it.obscuration, batoid.ObscNegation):
            ob = it.obscuration.original; oc, ic = True, False
        else:
            ob = it.obscuration; oc, ic = False, True
        rO = getattr(ob, "radius", None) or getattr(ob, "outer", np.nan)
        rI = getattr(ob, "inner", np.nan)
        for edge, rE, clr in [("outer", rO, oc), ("inner", rI, ic)]:
            if not np.isfinite(rE):
                continue
            ep = _edge_pupil_pts(xP, yP, xI, yI, rE)
            if ep is None or ep[0].size < 8:
                continue
            c = _fit_circle(*ep)
            if np.isfinite(c[1]):
                fits[(name, edge)] = dict(clear=clr, circ=c, ell=_fit_ellipse(*ep))
    return fits


fits_infocus = fit_edges(fiducial)
print(f"{'element':<16s}{'edge':<6s}  circle (xc, r)        ellipse (xc, a, b)")
for (name, edge), f in fits_infocus.items():
    xc, rc = f["circ"]; xe, a, b = f["ell"]
    tag = "ellipse" if abs(a - b) > 0.02 else "circular"
    print(f"{name:<16s}{edge:<6s}  ({xc:+6.2f},{rc:6.3f})     ({xe:+6.2f},{a:6.3f},{b:6.3f})  {tag}")


In [ ]:
# Compare circle vs ellipse mask agreement: matched config, and giant-intra
# (fixed nominal mask vs mask refit at the defocal configuration).
def eval_mask(u, v, fits, kind):
    cs, sn = np.cos(phi), np.sin(phi); keep = np.ones(len(u), bool)
    for f in fits.values():
        if kind == "circ":
            xc, rc = f["circ"]; cx, cy = xc * cs, xc * sn
            inside = ((u - cx) ** 2 + (v - cy) ** 2) <= rc ** 2
        else:
            xc, a, b = f["ell"]; cx, cy = xc * cs, xc * sn
            du, dv = u - cx, v - cy; e1 = du * cs + dv * sn; e2 = -du * sn + dv * cs
            inside = (e1 / a) ** 2 + (e2 / b) ** 2 <= 1
        keep &= inside if f["clear"] else ~inside
    return keep


def mask_agreement(optic, fits, kind):
    xP, yP, kept, _ = stop_boundary(optic, tx, ty)
    return (eval_mask(xP, yP, fits, kind) == kept).mean()


gi = build_case(fiducial, [("LSSTCamera", -giant_offset_mm)])
fits_gi = fit_edges(gi)
rows = [
    ("in-focus (matched)", "circle (ts_wep)",       mask_agreement(fiducial, fits_infocus, "circ")),
    ("in-focus (matched)", "ellipse",               mask_agreement(fiducial, fits_infocus, "ell")),
    ("giant intra",        "fixed circle (danish)", mask_agreement(gi, fits_infocus, "circ")),
    ("giant intra",        "fixed ellipse",         mask_agreement(gi, fits_infocus, "ell")),
    ("giant intra",        "defocal-refit circle",  mask_agreement(gi, fits_gi, "circ")),
    ("giant intra",        "defocal-refit ellipse", mask_agreement(gi, fits_gi, "ell")),
]
print(f"{'config':<20s}{'mask model':<24s}agreement")
for cfg, mdl, a in rows:
    print(f"{cfg:<20s}{mdl:<24s}{100*a:6.2f}%")

fig, ax = plt.subplots(figsize=(9.5, 4.6), constrained_layout=True)
colors = ["C0", "C1", "C3", "C4", "C2", "C5"]
ax.bar(range(len(rows)), [100 * a for *_, a in rows], color=colors)
ax.set_xticks(range(len(rows)))
ax.set_xticklabels([f"{c}\n{m}" for c, m, _ in rows], rotation=30, ha="right", fontsize=7)
ax.set_ylabel("mask vs batoid agreement [%]"); ax.set_ylim(92, 100)
ax.axhline(100 * rows[0][2], color="k", lw=0.6, ls=":")
ax.set_title("Circle vs ellipse edge model — ellipse does not help;\n"
             "defocal-refit (even a circle) recovers the giant-intra case")
ax.grid(alpha=0.3, axis="y")
plt.show()


<a id='telecentric'></a>
## Non-telecentricity in the Pupil / Donut Shape

The LSST chief ray is **not** normal to the focal plane off-axis: at the field edge its angle
of incidence is ~6°. A defocused donut is the pupil projected along that tilted chief ray, so
non-telecentricity shows up two ways: (i) the donut **shifts laterally** with the sign of
defocus — intra and extra move in opposite directions by `Δz·tan(AOI)` — and (ii) the pupil
is **foreshortened** by `cos(AOI)` along the tilt axis (a small, ~sub-percent anamorphic
squash). danish/ts_wep account for this through the field-angle-dependent off-axis model
(which carries the tilt) plus the per-donut centre (`dx, dy`) it fits. The plot below shows
the intra/extra batoid donuts at the `th_fov` field with their centroids and the in-focus
(telecentric) reference.

**Note — the centroid shift is *not* germane to the wavefront.** The absolute donut location
on the focal plane does not enter the Zernike estimate (danish fits the donut centre `dx, dy`
inside the postage stamp), so the telecentric astrometric shift shown above is harmless. What
*could* matter is the **pupil/donut shape**. We measured it: at 1.72° the donut is ~7%
elliptical (squashed along the field axis), but the pure cos(6.6°) foreshortening predicts
only ~0.3%. The ellipticity is dominated by the **field-dependent pupil mask** (the
obscuration circles shift off-axis → an asymmetric annulus), with a small wavefront
contribution; the geometric foreshortening is a negligible sub-effect (see the explanation of
danish's forward model below).

In [ ]:
# Non-telecentricity: chief-ray AOI, and the intra/extra donut centroid shift.
from scipy.ndimage import gaussian_filter

inst_fam = getTaskInstrument("LSSTCam", detector_name)
inst_fam.defocalOffset = fam_offset_mm * 1e-3
fl_m, pix_m = inst_fam.focalLength, inst_fam.pixelSize


def chief_ray_dir(th_deg):
    """Direction cosines of the pupil-centre (chief) ray at the detector."""
    r = batoid.RayVector.asPolar(optic=fiducial, wavelength=wavelength,
                                 theta_x=np.deg2rad(th_deg), theta_y=0, nrad=30, naz=120, inner=0)
    pr = r.copy().toCoordSys(fiducial.stopSurface.coordSys); fiducial.stopSurface.surface.intersect(pr)
    ic = int(np.argmin(np.hypot(pr.x, pr.y)))
    ch = r[ic:ic + 1].copy(); fiducial.trace(ch)
    return np.array([ch.vx[0], ch.vy[0], ch.vz[0]]), ch


def geom_donut_mm(defocal_mm, th_deg, npix=260, seeing=seeing_arcsec, osamp=3):
    """Batoid spot-diagram donut (seeing-convolved) on a detector mm grid; returns
    (image, centroid_mm, extent_mm)."""
    tel = build_case(fiducial, [("LSSTCamera", defocal_mm)])
    r = batoid.RayVector.asPolar(optic=tel, wavelength=wavelength,
                                 theta_x=np.deg2rad(th_deg), theta_y=0, nrad=300, naz=1800, inner=0)
    tel.trace(r); ok = ~r.vignetted & ~r.failed
    x, y = r.x[ok] * 1e3, r.y[ok] * 1e3                      # mm
    cx, cy = x.mean(), y.mean()
    halfmm = npix * (pix_m * 1e3) / 2; nb = npix * osamp
    H, _, _ = np.histogram2d(x, y, bins=nb,
                             range=[[cx - halfmm, cx + halfmm], [cy - halfmm, cy + halfmm]])
    H = H.T
    H = gaussian_filter(H, (seeing / 2.355) * np.radians(1 / 3600) * fl_m / (pix_m / osamp))
    img = H.reshape(npix, osamp, npix, osamp).sum(axis=(1, 3))
    return img, (cx, cy), [cx - halfmm, cx + halfmm, cy - halfmm, cy + halfmm]


v_chief, _ = chief_ray_dir(th_fov)
aoi = np.degrees(np.arccos(abs(v_chief[2])))
_, chief0 = chief_ray_dir(th_fov)                            # in-focus chief landing (telecentric ref)
x_ref = chief0.x[0] * 1e3

dE, cE, extE = geom_donut_mm(+fam_offset_mm, th_fov)
dI, cI, extI = geom_donut_mm(-fam_offset_mm, th_fov)
print(f"chief-ray AOI at {th_fov}° = {aoi:.2f}°  (telecentric ref x = {x_ref:.3f} mm)")
print(f"extra centroid x = {cE[0]:.3f} mm  (offset {1e3*(cE[0]-x_ref):+.0f} um)")
print(f"intra centroid x = {cI[0]:.3f} mm  (offset {1e3*(cI[0]-x_ref):+.0f} um)")
print(f"extra-intra centroid shift = {1e3*(cE[0]-cI[0]):+.0f} um  "
      f"(telecentric prediction 2*{fam_offset_mm}mm*tan({aoi:.1f}deg) = {1e3*2*fam_offset_mm*np.tan(np.deg2rad(aoi)):.0f} um)")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), constrained_layout=True)
for ax, img, c, ext, lab in [(axes[0], dI, cI, extI, "intra (camera −)"),
                             (axes[1], dE, cE, extE, "extra (camera +)")]:
    ax.imshow(img, origin="lower", cmap=cmap, extent=ext)
    ax.axvline(x_ref, color="cyan", lw=1, ls="--", label="telecentric ref")
    ax.plot(c[0], c[1], "r+", ms=12, mew=2, label="centroid")
    ax.set_title(f"{lab}\ncentroid x {1e3*(c[0]-x_ref):+.0f} µm from ref", fontsize=9)
    ax.set_xlabel("focal-plane x [mm]"); ax.set_ylabel("y [mm]"); ax.legend(fontsize=7, loc="upper right")
axes[2].axhline(0, color="k", lw=0.6)
axes[2].bar(["intra", "extra"], [1e3 * (cI[0] - x_ref), 1e3 * (cE[0] - x_ref)], color=["C0", "C1"])
axes[2].set_ylabel("centroid x − telecentric ref [µm]")
axes[2].set_title(f"Telecentric shift (chief AOI {aoi:.1f}°)\nopposite signs intra/extra", fontsize=9)
axes[2].grid(alpha=0.3, axis="y")
plt.show()


<a id='aperbias'></a>
## Synthetic Donut + ts_wep Fit — M1M3 Aperture-Update Bias

Reusable tooling: render a batoid spot-diagram donut (optionally clipping the pupil to a
chosen aperture annulus), apply `seeing_arcsec` Gaussian seeing, pixelate, scale to
`donut_peak_e` and add a sky pedestal + Poisson noise, then run the **paired ts_wep danish
fit** (`WfEstimator`, `jointFitPair=True`).

**Study:** draw the intra/extra FAM donuts with the **newly measured M1M3 aperture**
(`m1_inner_measured`, `m1_outer_measured` = 2.5833, 4.165 m) but **fit with the values
currently in use** (the design 2.558 / 4.18 m). The bias is the fitted-Zernike difference
versus drawing with the default aperture (so the frame offset between fitted and intrinsic
cancels). The aperture change is concentric, so the bias is expected in the **symmetric**
terms (defocus / spherical), not coma.

**Result.** The bias is small and **spherical-dominated**: ~+9 nm at Noll 11 (robust, stable across ray counts), ~+7 nm Z8, **~3 nm RMS** overall — consistent with a concentric aperture change. *Caveat:* the aperture is imposed as a hard radial clip on the traced rays, so the thin clipped shell (0.4 % outer, 0.6 % inner of the radius) is finely sampled; the **spherical term is stable**, but the small **coma (Z8) component is sensitive to the ray sampling** (`nrad`) — treat it as order-of-magnitude. Use `nrad`≈320–500; avoid sparse grids.

In [ ]:
# --- Tooling: batoid donut (optional aperture clip) -> seeing -> pixelate -> noise; ts_wep fit ---
from scipy.ndimage import gaussian_filter
from lsst.ts.wep.image import Image
from lsst.ts.wep.estimation import WfEstimator
from lsst.ts.wep.utils import DefocalType, BandLabel

donut_inst = getTaskInstrument("LSSTCam", detector_name)
donut_inst.defocalOffset = fam_offset_mm * 1e-3


def synth_donut(defocal_mm, th_deg, aperture=None, npix=185, seeing=seeing_arcsec,
                peak=donut_peak_e, sky=donut_sky_e, seed=1, osamp=4):
    """ts_wep-ready donut image from a batoid spot diagram. aperture=(r_in, r_out) in m clips
    the pupil (None = batoid's own vignetting)."""
    rng = np.random.default_rng(seed)
    tel = build_case(fiducial, [("LSSTCamera", defocal_mm)])
    r = batoid.RayVector.asPolar(optic=tel, wavelength=wavelength,
                                 theta_x=np.deg2rad(th_deg), theta_y=0, nrad=320, naz=1900, inner=0)
    pr = r.copy().toCoordSys(tel.stopSurface.coordSys); tel.stopSurface.surface.intersect(pr)
    rr = np.hypot(pr.x, pr.y)
    tel.trace(r); ok = (~r.vignetted) & (~r.failed)
    if aperture is not None:
        ok &= (rr >= aperture[0]) & (rr <= aperture[1])
    x, y = r.x[ok], r.y[ok]; cx, cy = x.mean(), y.mean()
    pix_m = donut_inst.pixelSize; half = npix * pix_m / 2; nb = npix * osamp
    H, _, _ = np.histogram2d(x, y, bins=nb, range=[[cx - half, cx + half], [cy - half, cy + half]])
    H = H.T
    H = gaussian_filter(H, (seeing / 2.355) * np.radians(1 / 3600) * donut_inst.focalLength / (pix_m / osamp))
    img = H.reshape(npix, osamp, npix, osamp).sum(axis=(1, 3)); img *= peak / img.max()
    return rng.poisson(img + sky).astype(float)


def joint_fit(imgE, imgI, th_deg, instrument):
    """Paired danish fit; returns (zk_um, history)."""
    IE = Image(imgE, (th_deg, 0.0), DefocalType.Extra, BandLabel.REF)
    II = Image(imgI, (th_deg, 0.0), DefocalType.Intra, BandLabel.REF)
    wfe = WfEstimator(algoName="danish", algoConfig={"jointFitPair": True}, instConfig=instrument,
                      nollIndices=np.array(fit_noll), startWithIntrinsic=True, units="um", saveHistory=True)
    zk = np.asarray(wfe.estimateZk(IE, II)[0]).ravel()
    return zk, wfe.history


ap_default = (donut_inst.radius * donut_inst.obscuration, donut_inst.radius)   # design 2.558, 4.180
ap_measured = (m1_inner_measured, m1_outer_measured)                           # measured 2.5833, 4.165

zk_def, _ = joint_fit(synth_donut(+fam_offset_mm, th_fov, ap_default, seed=1),
                      synth_donut(-fam_offset_mm, th_fov, ap_default, seed=2), th_fov, donut_inst)
zk_meas, hist = joint_fit(synth_donut(+fam_offset_mm, th_fov, ap_measured, seed=1),
                          synth_donut(-fam_offset_mm, th_fov, ap_measured, seed=2), th_fov, donut_inst)
bias = zk_meas - zk_def
jmax = fit_noll[int(np.argmax(np.abs(bias)))]
print(f"M1M3 aperture-update bias (draw measured, fit default), FAM {th_fov}°:")
print(f"  RMS = {1e3*np.sqrt(np.mean(bias**2)):.0f} nm,  max = {1e3*np.max(np.abs(bias)):.0f} nm at Noll {jmax}")

fig, ax = plt.subplots(figsize=(8.5, 4), constrained_layout=True)
ax.bar(fit_noll, 1e3 * bias, color="C3")
ax.set_xticks(fit_noll); ax.set_xlabel("Noll index"); ax.set_ylabel("Zernike bias [nm]")
ax.set_title(f"M1M3 aperture-update bias — draw ({ap_measured[0]:.4f}, {ap_measured[1]:.3f}) m, "
             f"fit ({ap_default[0]:.3f}, {ap_default[1]:.2f}) m;  FAM {th_fov}°")
ax.grid(alpha=0.3, axis="y"); plt.show()

# data / model / residual for the measured-aperture fit
fig, axs = plt.subplots(2, 3, figsize=(11, 7.4), constrained_layout=True)
for row, side in enumerate(["extra", "intra"]):
    data = hist[side]["image"]; model = hist[side]["model"]; resid = data - model
    for ax, arr, ttl, cm in [(axs[row, 0], data, "data", cmap), (axs[row, 1], model, "model", cmap),
                             (axs[row, 2], resid, "residual", "RdBu_r")]:
        if ttl == "residual":
            v = np.nanpercentile(np.abs(arr), 99.5)
            im = ax.imshow(arr, origin="lower", cmap=cm, vmin=-v, vmax=v)
        else:
            im = ax.imshow(arr, origin="lower", cmap=cm)
        ax.set_title(f"{side} — {ttl}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
fig.suptitle(f"Measured-aperture FAM donut joint fit (data/model/residual) — {th_fov}°", fontsize=12)
plt.show()


<a id='wfsgrid'></a>
## WFS Pupil: Danish vs Batoid vs Difference, 1.60–1.725°

Direct Danish / Batoid / difference comparison of the donut (pupil) flux pattern for the
**R04 corner WFS**, swept over field radius 1.60–1.725° (in 0.025° steps, along the R04
azimuth), on both sides of focus. The corner WFS defocus is a **CCD piston** (the
`batoidOffsetOptic` is the Detector), so the batoid donut offsets the Detector to match what
ts_wep/danish assume. danish is rendered with `z_fit=0` (off-axis `z_ref` only). Two figures:
intra-focal and extra-focal.

In [ ]:
# WFS (R04) Danish/Batoid/Difference pupil grid vs field angle, both sides of focus.
import lsst.geom

def raft_azimuth_deg(det):
    d = camera[det]; c = d.getBBox().getCenter()
    fa = d.getTransform(PIXELS, FIELD_ANGLE).applyForward(lsst.geom.Point2D(c))
    return np.degrees(np.arctan2(fa.getY(), fa.getX()))


def pupil_triplet(detname, dtype, th_deg, az_deg, defocal_mm, half, npix=181):
    """Unit-sum danish (z_fit=0) and batoid donuts at field radius th_deg along az_deg.
    Offsets inst.batoidOffsetOptic (Detector for WFS, Camera for FAM) to match ts_wep."""
    inst = getTaskInstrument("LSSTCam", detname); inst.defocalOffset = defocal_mm * 1e-3
    tx = th_deg * np.cos(np.deg2rad(az_deg)); ty = th_deg * np.sin(np.deg2rad(az_deg))
    sign = +1 if dtype == DefocalType.Extra else -1
    zr = inst.getOffAxisCoeff(tx, ty, dtype, "r", nollIndicesModel=np.arange(79),
                              nollIndicesIntr=np.arange(4, 23))
    ps = 2 * half * 1e-3 / npix
    fac = danish.DonutFactory(R_outer=inst.radius, R_inner=inst.radius * inst.obscuration,
                              mask_params=inst.maskParams, focal_length=inst.focalLength, pixel_scale=ps)
    D = danish.SingleDonutModel(fac, z_ref=zr, z_terms=(), thx=np.deg2rad(tx), thy=np.deg2rad(ty),
                                npix=npix).model(flux=1.0, dx=0, dy=0, fwhm=0.3, z_fit=())
    D = D / D.sum()
    tel = inst.getBatoidModel().withLocallyShiftedOptic(inst.batoidOffsetOptic, [0, 0, sign * defocal_mm * 1e-3])
    r = batoid.RayVector.asPolar(optic=tel, wavelength=wavelength, theta_x=np.deg2rad(tx),
                                 theta_y=np.deg2rad(ty), nrad=200, naz=1200, inner=0)
    tel.trace(r); ok = ~r.vignetted & ~r.failed
    x, y = r.x[ok] * 1e3, r.y[ok] * 1e3; cx, cy = x.mean(), y.mean()
    H, _, _ = np.histogram2d(x, y, bins=npix, range=[[cx - half, cx + half], [cy - half, cy + half]])
    B = H.T / H.sum()
    return B, D


def pupil_grid(detname, az_deg, defocal_mm, half, dtype, label):
    ext = [-half, half, -half, half]
    fig, axes = plt.subplots(len(grid_angles), 3, figsize=(8.5, 2.7 * len(grid_angles)),
                             constrained_layout=True)
    for i, th in enumerate(grid_angles):
        B, D = pupil_triplet(detname, dtype, th, az_deg, defocal_mm, half)
        vmax = max(B.max(), D.max()); diff = B - D; v = np.nanpercentile(np.abs(diff), 99.5)
        corr = np.corrcoef(B.ravel(), D.ravel())[0, 1]
        axes[i, 0].imshow(D, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
        axes[i, 1].imshow(B, origin="lower", cmap=cmap, vmin=0, vmax=vmax, extent=ext)
        axes[i, 2].imshow(diff, origin="lower", cmap="RdBu_r", vmin=-v, vmax=v, extent=ext)
        axes[i, 0].set_ylabel(f"{th:.3f} deg", fontsize=9)
        axes[i, 2].text(0.97, 0.05, f"corr {corr:.3f}", transform=axes[i, 2].transAxes,
                        ha="right", fontsize=7, color="k")
        for a in axes[i]:
            a.set_xticks([]); a.set_yticks([])
    axes[0, 0].set_title("danish (z_fit=0)"); axes[0, 1].set_title("batoid"); axes[0, 2].set_title("batoid − danish")
    fig.suptitle(label, fontsize=12)
    plt.show()


grid_angles = np.array([1.600, 1.625, 1.650, 1.675, 1.700, 1.725])
az_R04 = raft_azimuth_deg("R04_SW0")
print(f"R04 azimuth = {az_R04:.1f}°; WFS defocus = Detector piston {wfs_offset_mm} mm")
pupil_grid("R04_SW1", az_R04, wfs_offset_mm, half_mm_fam, DefocalType.Intra,
           f"WFS R04 — INTRA (Detector −{wfs_offset_mm} mm), az {az_R04:.0f}°")
pupil_grid("R04_SW0", az_R04, wfs_offset_mm, half_mm_fam, DefocalType.Extra,
           f"WFS R04 — EXTRA (Detector +{wfs_offset_mm} mm), az {az_R04:.0f}°")


<a id='famgrid'></a>
## FAM Pupil: Danish vs Batoid vs Difference, 1.60–1.725°

Same Danish / Batoid / difference comparison for the **FAM** case (camera-hexapod defocus),
swept 1.60–1.725° along the **R03** azimuth, intra and extra. Here the `batoidOffsetOptic` is
the **Camera**, so the batoid donut pistons the camera (filter/L1/L2 move with it). Compare to
the WFS grid above: the FAM intra side shows the camera-borne (filter) departure growing near
the field edge, which the WFS case (camera fixed) does not.

In [ ]:
# FAM (R03 azimuth) Danish/Batoid/Difference pupil grid, both sides of focus.
az_R03 = raft_azimuth_deg("R03_S11")
print(f"R03 azimuth = {az_R03:.1f}°; FAM defocus = Camera piston {fam_offset_mm} mm")
pupil_grid("R03_S11", az_R03, fam_offset_mm, half_mm_fam, DefocalType.Intra,
           f"FAM — INTRA (Camera −{fam_offset_mm} mm), az {az_R03:.0f}° (R03)")
pupil_grid("R03_S11", az_R03, fam_offset_mm, half_mm_fam, DefocalType.Extra,
           f"FAM — EXTRA (Camera +{fam_offset_mm} mm), az {az_R03:.0f}° (R03)")


<a id='filteronset'></a>
## Intra-focal Filter-Vignetting Onset

Where does the camera-borne filter start clipping the pupil? Per-surface vignetting
attribution (batoid `traceFull`) of the fraction of pupil rays first vignetted by the filter,
vs field radius, for the intra-focal FAM and giant (camera-hexapod) configurations. The filter
turns on at ~1.73° — just **outside** the 1.725° science cut — so it is negligible within the
cut and grows beyond it (much faster for the 8 mm giant defocus than for FAM ±1.5 mm).

In [ ]:
# Filter-vignetting fraction vs field radius (intra-focal), camera-hexapod defocus.
def filter_fraction(defocal_mm, th_deg, az_deg=0.0, nrad=150):
    inst = getTaskInstrument("LSSTCam", detector_name); inst.defocalOffset = abs(defocal_mm) * 1e-3
    tel = inst.getBatoidModel().withLocallyShiftedOptic("LSSTCamera", [0, 0, -abs(defocal_mm) * 1e-3])  # intra
    tx = th_deg * np.cos(np.deg2rad(az_deg)); ty = th_deg * np.sin(np.deg2rad(az_deg))
    r = batoid.RayVector.asPolar(optic=tel, wavelength=wavelength, theta_x=np.deg2rad(tx),
                                 theta_y=np.deg2rad(ty), nrad=nrad, naz=nrad * 6, inner=0)
    first = np.array([""] * len(r.x), dtype=object); vprev = np.zeros(len(r.x), bool)
    for nm, d in tel.traceFull(r).items():
        vo = d["out"].vignetted; nw = vo & ~vprev; first[nw] = nm; vprev = vprev | vo
    return 100 * np.sum(first == "Filter_entrance") / len(r.x)


ths = np.arange(1.50, 1.86, 0.02)
frac_fam = [filter_fraction(fam_offset_mm, t) for t in ths]
frac_giant = [filter_fraction(defocus_mm, t) for t in ths]
fig, ax = plt.subplots(figsize=(8, 4.4), constrained_layout=True)
ax.plot(ths, frac_fam, "o-", label=f"FAM intra (camera −{fam_offset_mm} mm)")
ax.plot(ths, frac_giant, "s-", label=f"giant intra (camera −{defocus_mm} mm)")
ax.axvline(1.725, color="k", ls="--", lw=1, label="1.725° science cut")
ax.set_xlabel("field radius [deg]"); ax.set_ylabel("rays vignetted by filter [%]")
ax.set_title("Intra-focal filter-vignetting onset (~1.73°, just outside the cut)")
ax.grid(alpha=0.3); ax.legend()
plt.show()
onset = ths[np.argmax(np.array(frac_fam) > 0.05)]
print(f"FAM intra filter onset ≈ {onset:.2f}°  (filter fraction at 1.72° = {filter_fraction(fam_offset_mm, 1.72):.2f}%)")


<a id='biassim'></a>
## Pupil-Mismatch Zernike Bias from Simulation (summary)

How much do the pupil-model imperfections actually bias the fitted wavefront? Using the §13
tooling (batoid donut → seeing → noise → ts_wep paired danish fit), we quantified two effects
at FAM 1.72°. Both are **small** at the science cut:

- **M1M3 aperture update** (§13): drawing with the measured aperture (2.5833 / 4.165 m) and
  fitting with the default → **~3 nm RMS, ~9 nm spherical (Noll 11)**.
- **Filter mismatch** (this section): the filter is absent from danish's mask but present in
  the batoid "data". Since the filter only vignettes beyond ~1.73° (previous section), its
  wavefront bias is ~0 at the 1.72° cut and grows just outside it. We measure the **coma (Z8)
  excess** relative to the 1.72° (filter-free) baseline, which cancels the constant
  fitted-vs-intrinsic frame offset.

*Caveat:* absolute fitted-vs-intrinsic comparisons carry a ~12 nm frame offset (fit-frame vs
CCS), so biases are reported as **differences** of two fits.

In [ ]:
# Filter-mismatch wavefront bias vs field: fit batoid "data" (filter present) with danish.
def fam_fit_coma(th_deg, seed=1):
    """Joint danish fit of batoid FAM donuts (full optics, filter present) -> fitted minus
    intrinsic Zernikes (um). Z8 (Noll 8) is the coma the asymmetric filter crescent biases."""
    IE = Image(synth_donut(+fam_offset_mm, th_deg, None, seed=seed), (th_deg, 0.0),
               DefocalType.Extra, BandLabel.REF)
    II = Image(synth_donut(-fam_offset_mm, th_deg, None, seed=seed + 1), (th_deg, 0.0),
               DefocalType.Intra, BandLabel.REF)
    wfe = WfEstimator(algoName="danish", algoConfig={"jointFitPair": True}, instConfig=donut_inst,
                      nollIndices=np.array(fit_noll), startWithIntrinsic=True, units="um", saveHistory=True)
    zk = np.asarray(wfe.estimateZk(IE, II)[0]).ravel()
    intr = np.asarray(donut_inst.getIntrinsicZernikes(th_deg, 0.0, defocalType=None,
                                                      nollIndices=np.array(fit_noll))).ravel() * 1e6
    return zk - intr


bias_fields = [1.72, 1.76, 1.80]
diffs = {t: fam_fit_coma(t) for t in bias_fields}
base = diffs[1.72]
print("Filter-mismatch bias vs the 1.72° (filter-free) baseline — Noll 8 (coma):")
for t in bias_fields:
    z8 = 1e3 * (diffs[t][fit_noll.index(8)] - base[fit_noll.index(8)])
    rms = 1e3 * np.sqrt(np.mean((diffs[t] - base) ** 2))
    print(f"  field {t:.2f}°: filter {filter_fraction(fam_offset_mm, t):4.1f}%  ->  ΔZ8 = {z8:+5.0f} nm, RMS Δ = {rms:4.0f} nm")
print("=> filter bias is ~0 at the 1.725° cut and reaches only a few tens of nm just beyond it.")
